# swapface sur ColabComfyUI et ReActor tournent **ici**, sur le GPU Colab. L'interface s'affiche danston navigateur, et ton Mac peut piloter le rendu par l'API grace au tunnel.Ordre des cellules : carte, disque persistant, installation, modeles, verification,demarrage, tunnel.Rappel mesure sur un Mac M5 pour comparaison : 96,7 ms par image sans restauration,452 ms avec CodeFormer. En dessous de deux minutes de video, Colab ne vaut pas ledetour une fois comptes l'installation et le televersement.

## 1. Quelle carte as-tu obtenue

In [ ]:
# A100 et L4 valent le detour. Sur T4, l'ecart avec un Mac M5 patche est modeste.# Un TPU ne sert a rien ici : onnxruntime n'a pas de provider TPU et ReActor# repose sur onnxruntime plus PyTorch CUDA.!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "AUCUN GPU : va dans Execution, Modifier le type d'execution"!free -g | head -2!df -h /content | tail -1

## 2. Disque persistant, facultatif mais recommandeSans Drive, les 1,6 Go de modeles se retelechargent a chaque session. Avec Drive,une fois suffit. Le montage demande une autorisation Google.Passe cette cellule si tu preferes tout jeter en fin de session.

In [ ]:
PERSISTER = True  # False pour tout garder dans /content, efface a la deconnexionfrom pathlib import Pathif PERSISTER:    from google.colab import drive    drive.mount('/content/drive')    MODELES_PERSISTANTS = Path('/content/drive/MyDrive/swapface/models')    MODELES_PERSISTANTS.mkdir(parents=True, exist_ok=True)else:    MODELES_PERSISTANTS = Noneprint('modeles persistants :', MODELES_PERSISTANTS)

## 3. Le depot swapfaceOn reprend `verif_modeles.py`, `telecharger_modeles.py` et `workflow.json` plutotque de les recopier ici. Si le depot est prive, la cellule demande un jeton.

In [ ]:
import subprocess, getpassfrom pathlib import PathDEPOT = Path('/content/swapface')URL = 'https://github.com/kofekod23/swapface.git'if not DEPOT.exists():    resultat = subprocess.run(['git', 'clone', '--depth', '1', URL, str(DEPOT)],                              capture_output=True, text=True)    if resultat.returncode != 0:        print("Clone anonyme refuse, le depot doit etre prive.")        jeton = getpass.getpass('Jeton GitHub (masque, non conserve) : ')        url_authentifiee = URL.replace('https://', f'https://x-access-token:{jeton}@')        subprocess.run(['git', 'clone', '--depth', '1', url_authentifiee, str(DEPOT)],                       check=True, capture_output=True)        # On remet une remote sans le jeton.        subprocess.run(['git', '-C', str(DEPOT), 'remote', 'set-url', 'origin', URL], check=True)        del jetonprint('contenu :', sorted(p.name for p in DEPOT.iterdir() if p.is_file()))

## 4. Installation de ComfyUI, ReActor et VideoHelperSuite

In [ ]:
%%timeimport os, subprocessfrom pathlib import PathCOMFY = Path('/content/ComfyUI')NOEUDS = COMFY / 'custom_nodes'def executer(commande, dossier=None):    resultat = subprocess.run(commande, cwd=dossier, capture_output=True, text=True)    if resultat.returncode != 0:        print(resultat.stdout[-2000:]); print(resultat.stderr[-2000:])        raise RuntimeError(' '.join(commande))if not COMFY.exists():    executer(['git', 'clone', '--depth', '1',              'https://github.com/Comfy-Org/ComfyUI.git', str(COMFY)])    executer(['git', 'clone', '--depth', '1', '-b', 'main',              'https://github.com/Gourieff/ComfyUI-ReActor.git',              str(NOEUDS / 'ComfyUI-ReActor')])    executer(['git', 'clone', '--depth', '1',              'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',              str(NOEUDS / 'ComfyUI-VideoHelperSuite')])# torch est deja present et compile pour CUDA sur Colab, pip le laissera tranquille.!pip install -q -r {COMFY}/requirements.txt!pip install -q -r {NOEUDS}/ComfyUI-VideoHelperSuite/requirements.txt# install.py de ReActor a besoin de pkg_resources ou de son repli.!pip install -q importlib_metadataimport torchprint('CUDA disponible :', torch.cuda.is_available(),      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun')

## 5. Modeles`install.py` de ReActor ne pose que `inswapper_128.onnx`, et il choisit`onnxruntime-gpu` tout seul puisqu'il voit CUDA. Le reste vient de`telecharger_modeles.py`.Si tu as monte Drive, le dossier des modeles y est deplace puis relie par un liensymbolique : la session suivante ne retelecharge rien.

In [ ]:
%%timeimport os, shutil, subprocessfrom pathlib import PathMODELES = COMFY / 'models'# Lien vers Drive avant tout telechargement, pour que tout atterrisse au bon endroit.if MODELES_PERSISTANTS is not None and not MODELES.is_symlink():    if MODELES.exists():        for element in MODELES.iterdir():            cible = MODELES_PERSISTANTS / element.name            if not cible.exists():                shutil.move(str(element), str(cible))        shutil.rmtree(MODELES)    MODELES.symlink_to(MODELES_PERSISTANTS)    print('models relie a', MODELES_PERSISTANTS)subprocess.run(['python', 'install.py'], cwd=str(NOEUDS / 'ComfyUI-ReActor'), check=True)# telecharger_modeles.py attend d'etre a cote d'un dossier ComfyUI.lien = Path('/content/swapface/ComfyUI')if not lien.exists():    lien.symlink_to(COMFY)subprocess.run(['python', 'telecharger_modeles.py', '--hyperswap'],               cwd='/content/swapface', check=True)

## 6. Verification des empreintes

In [ ]:
!python /content/swapface/verif_modeles.py {COMFY}

## 7. Demarrage de ComfyUISur CUDA, `CUDAExecutionProvider` prend l'integralite du graphe onnx, la ou CoreMLn'en prend qu'une partition sur Mac. Rien a regler.

In [ ]:
import subprocess, time, refrom pathlib import PathJOURNAL = Path('/content/comfyui.log')PORT = 8188serveur = subprocess.Popen(    ['python', 'main.py', '--port', str(PORT)],    cwd=str(COMFY),    stdout=JOURNAL.open('w'), stderr=subprocess.STDOUT,)for _ in range(180):    texte = JOURNAL.read_text(errors='ignore')    if 'Starting server' in texte:        break    if serveur.poll() is not None:        print(texte[-3000:]); raise RuntimeError('ComfyUI s\'est arrete')    time.sleep(1)else:    raise TimeoutError('demarrage trop long, regarde /content/comfyui.log')peripherique = re.search(r'Device:\s*(\S+)', texte)print('peripherique :', peripherique.group(1) if peripherique else 'inconnu')print('serveur pret sur le port', PORT)

## 8. TunnelDeux sorties. Le proxy Colab suffit si tu veux juste l'interface dans tonnavigateur : aucune installation, aucun tiers, mais l'URL est liee a ton cookieGoogle et un script exterieur ne peut pas s'en servir.Le tunnel cloudflared donne une URL publique utilisable par `piloter.py` depuis tonMac. Pas de compte, pas de jeton. Limites annoncees par Cloudflare : 200 requetessimultanees, pas de Server-Sent Events, service sans garantie de disponibilite.ComfyUI passe par WebSocket, donc cela convient.

In [ ]:
# Voie A, interface seulement, dans ce navigateur.from google.colab.output import eval_jsprint('interface Colab :', eval_js(f'google.colab.kernel.proxyPort({PORT})'))

In [ ]:
# Voie B, URL publique, utilisable depuis ton Mac.import reimport statimport subprocessimport timeimport urllib.requestfrom pathlib import PathBINAIRE = Path('/content/cloudflared')SOURCE = ('https://github.com/cloudflare/cloudflared/releases/latest/download/'          'cloudflared-linux-amd64')if not BINAIRE.exists():    urllib.request.urlretrieve(SOURCE, BINAIRE)    BINAIRE.chmod(BINAIRE.stat().st_mode | stat.S_IXUSR)    print('cloudflared installe')JOURNAL_TUNNEL = Path('/content/cloudflared.log')tunnel = subprocess.Popen(    [str(BINAIRE), 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],    stdout=JOURNAL_TUNNEL.open('w'), stderr=subprocess.STDOUT,)adresse = Nonefor _ in range(60):    trouve = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',                       JOURNAL_TUNNEL.read_text(errors='ignore'))    if trouve:        adresse = trouve.group(0)        break    time.sleep(1)if not adresse:    print(JOURNAL_TUNNEL.read_text()[-2000:])    raise TimeoutError("pas d'URL de tunnel")print('URL publique :', adresse)print()print('Sur ton Mac, dans ~/Applications/claude/swapface :')print(f'  python3 piloter.py --serveur {adresse} \\')print('                     --video source.mp4 --visage mon_visage.jpg \\')print('                     --images 0 --sortie rendu.mp4')

## 9. ArretColab facture tant que la session tourne. Arrete-la quand tu as recupere ton rendu :menu Execution, puis Interrompre l'execution, ou la cellule ci-dessous.

In [ ]:
for processus, nom in ((tunnel, 'tunnel'), (serveur, 'ComfyUI')):    try:        processus.terminate(); processus.wait(timeout=20); print(nom, 'arrete')    except Exception as erreur:        print(nom, ':', erreur)